In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns

In [2]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [3]:
# 소수점 아래 10자리까지 출력
pd.options.display.float_format = '{:.10f}'.format

In [5]:
ns = 1e-9

xDim = 6
yDim = 6

In [7]:
benchmark = 'pagerank'
trace_file = f"./{benchmark}_mesh_6x6_memory_trace.csv"

df = pd.read_csv(trace_file, skipinitialspace=True)
df

,time,addr,pc,owner,rw,rdma,vector,type,chiplet,data_id,is_ptw
0,0.0006897310,2163957832,16806144,35,0,1,0,2,0,-1,1
1,0.0006897350,2163957824,16806152,35,0,1,0,0,0,-1,1
2,0.0006897860,2163957832,16806144,35,0,1,0,2,6,-1,1
3,0.0006897870,2163957832,16806144,35,0,1,0,2,1,-1,1
4,0.0006897900,2163957824,16806152,35,0,1,0,0,6,-1,1
...,...,...,...,...,...,...,...,...,...,...,...
423819,0.0007187740,2147561408,16806916,1,1,1,1,4,35,1,0
423820,0.0007187760,2147561408,16806916,1,1,1,1,4,35,1,0
423821,0.0007195260,2147561344,16806916,1,1,1,1,4,35,1,0
423822,0.0007195720,2147561408,16806916,1,1,1,1,4,35,1,0


In [6]:
def process_benchmark(benchmark, df):
    print(f"[{benchmark}] Analyzing PC stats for L2 Miss (RDMA) per Owner...")
    
    try:
        # 'type' 컬럼을 추가로 로드하여 L2 Miss 여부를 확인합니다.
        df = pd.read_csv(trace_file, skipinitialspace=True, usecols=['pc', 'addr', 'chiplet', 'rdma', 'owner', 'rw', 'type'])
        
        # 1. rdma == True 이면서 L2 Miss (type == 2 or 5) 인 경우만 필터링
        df = df[(df['rdma'] == True) & (df['type'].isin([2, 5]))].copy()

        if df.empty:
            return None

        src_x, src_y = df['chiplet'] % xDim, df['chiplet'] // xDim
        dst_x, dst_y = df['owner']   % xDim, df['owner']   // xDim
        df['hop_count'] = (src_x - dst_x).abs() + (src_y - dst_y).abs()

        # 2. 캐시 라인(addr)별 평균 요청 칩렛 개수 (address-equal weight: 모든 addr 동등 가중)
        addr_chiplets = df.groupby(['pc', 'addr'])['chiplet'].nunique().reset_index(name='chiplet_count')
        avg_chiplets_per_addr = addr_chiplets.groupby('pc')['chiplet_count'].mean().reset_index(name='avg_requestor_chiplets')

        # 3. 'owner'(데이터의 홈 노드)와 'pc'별로 그룹화하여 통계 추출
        stats = df.groupby(['owner', 'pc']).agg(
            access_count=('addr', 'count'),
            unique_addrs=('addr', 'nunique'),
            read_count=('rw', lambda x: (x == 0).sum()),
            write_count=('rw', lambda x: (x == 1).sum()),
            avg_hop=('hop_count', 'mean'),
        ).reset_index()
        stats['access_per_addr'] = stats['access_count'] / stats['unique_addrs']

        # 4. 같은 PC에 대해 여러 owner들에 대한 stat의 평균을 계산
        pc_agg = stats.groupby('pc').agg(
            avg_unique_addrs_per_owner=('unique_addrs', 'mean'),
            avg_read_count_per_owner=('read_count', 'mean'),
            avg_write_count_per_owner=('write_count', 'mean'),
            avg_access_per_addr_per_owner=('access_per_addr', 'mean'),
            avg_hop_count_per_owner=('avg_hop', 'mean'),
            total_access_count=('access_count', 'sum')
        ).reset_index()

        pc_agg = pd.merge(pc_agg, avg_chiplets_per_addr, on='pc')

        total_unique_addrs_per_pc = df.groupby('pc')['addr'].nunique().reset_index(name='total_unique_addrs')
        pc_agg = pd.merge(pc_agg, total_unique_addrs_per_pc, on='pc')

        # Read/Write 비율 계산 (분모를 avg_read + avg_write로 잡아 avg_access 없이도 동일한 값)
        rw_sum = pc_agg['avg_read_count_per_owner'] + pc_agg['avg_write_count_per_owner']
        pc_agg['read_ratio_%']  = (pc_agg['avg_read_count_per_owner']  / rw_sum * 100).round(2).fillna(0)
        pc_agg['write_ratio_%'] = (pc_agg['avg_write_count_per_owner'] / rw_sum * 100).round(2).fillna(0)
        pc_agg['avg_access_per_addr_per_owner'] = pc_agg['avg_access_per_addr_per_owner'].round(2)
        pc_agg['avg_hop_count_per_owner'] = pc_agg['avg_hop_count_per_owner'].round(2)

        pc_agg.insert(0, 'benchmark', benchmark)

        column_order = [
            'benchmark', 'pc',
            'total_access_count', 'total_unique_addrs',
            'avg_unique_addrs_per_owner',
            'avg_access_per_addr_per_owner',
            'avg_requestor_chiplets', 'avg_hop_count_per_owner',
            'read_ratio_%', 'write_ratio_%',
        ]
        pc_agg = pc_agg[column_order]

        # 정렬: 총 접근 횟수 기준 내림차순
        pc_agg = pc_agg.sort_values(by='total_access_count', ascending=False)

        return pc_agg

    except Exception as e:
        print(f"[{benchmark}] Error: {e}")
        return None

In [8]:
pc_agg_df = process_benchmark(benchmark, df)

[pagerank] Analyzing PC stats for L2 Miss (RDMA) per Owner...


In [9]:
pc_agg_df

,benchmark,pc,total_access_count,total_unique_addrs,avg_unique_addrs_per_owner,avg_access_per_addr_per_owner,avg_requestor_chiplets,avg_hop_count_per_owner,read_ratio_%,write_ratio_%
7,pagerank,16806484,3189,126,63.0000000000,25.3300000000,25.3095238095,4.8900000000,100.0000000000,0.0000000000
8,pagerank,16806488,1814,1067,34.4193548387,1.4100000000,1.7000937207,2.2200000000,100.0000000000,0.0000000000
4,pagerank,16806424,944,940,31.3333333333,1.0000000000,1.0042553191,1.8600000000,100.0000000000,0.0000000000
6,pagerank,16806480,344,103,51.5000000000,3.5000000000,3.3398058252,4.5600000000,100.0000000000,0.0000000000
2,pagerank,16806304,226,128,64.0000000000,1.7700000000,1.7656250000,5.0800000000,30.9700000000,69.0300000000
3,pagerank,16806312,160,126,42.0000000000,1.1800000000,1.2698412698,5.8200000000,100.0000000000,0.0000000000
5,pagerank,16806472,86,57,28.5000000000,1.5200000000,1.5087719298,5.2300000000,100.0000000000,0.0000000000
0,pagerank,16806144,70,2,1.0000000000,35.0000000000,35.0000000000,5.1400000000,100.0000000000,0.0000000000
1,pagerank,16806152,35,1,1.0000000000,35.0000000000,35.0000000000,5.1400000000,100.0000000000,0.0000000000


In [10]:
pc_agg_df['pc']

7    16806484
8    16806488
4    16806424
6    16806480
2    16806304
3    16806312
5    16806472
0    16806144
1    16806152
Name: pc, dtype: int64